# Chunking strategy test on your own documents
Compares 8 chunking strategies with the same embedding model (harrier-oss-v1-0.6b) and the same questions, so the only thing that changes is how the documents are cut.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

**You need:**
- Your **documents** as `.md`, `.txt`, or `.pdf`. Markdown headings (`#`, `##`, `###`) let the structure-aware strategies use your sections; PDFs lose headings when converted to text.
- An **`eval.csv`** with columns `query` and `answer_text`. `answer_text` is a sentence or short phrase **copied exactly** from your document that answers the question.

**Why `answer_text` and not a chunk id?** Chunk ids change every time you re-chunk, so they can't be compared across strategies. Instead, a result counts as correct if the returned chunk *contains* the answer text (exact, or a 90%+ fuzzy match).

No files yet? Skip the upload cell and run the demo cell.

In [ ]:
!pip install -q -U sentence-transformers transformers langchain-text-splitters rapidfuzz pypdf

## 1. Upload your documents and eval.csv

In [ ]:
import os, shutil
from google.colab import files
os.makedirs("docs", exist_ok=True)
uploaded = files.upload()   # select your documents (.md / .txt / .pdf) AND eval.csv
for name in uploaded:
    if name != "eval.csv":
        shutil.move(name, os.path.join("docs", name))
print("documents:", os.listdir("docs"))

## 1b. (Optional) Demo data: a Machine Learning handbook
One markdown document (8 chapters, 50 sections) and 100 questions. It is small, so use it to check the notebook works; make real decisions on your own, larger documents.

In [ ]:
import os, re, pandas as pd

# 50 textbook-style ML passages (written for this test), 2 questions each
DATA = [
    ('c01', 'The bias-variance tradeoff describes two sources of prediction error. A very simple model has high bias: it underfits because it cannot capture the true pattern. A very flexible model has high variance: its predictions change a lot with small changes in the training set. Good models balance the two so that error on unseen data is lowest.', ['why do a model that is too simple and one that is too complex both perform badly?', 'what is the tradeoff between underfitting and overfitting called']),
    ('c02', 'Overfitting happens when a model memorizes noise in the training data instead of learning general patterns, so training accuracy is high but test accuracy is poor. Common remedies include collecting more data, simplifying the model, and adding regularization, which penalizes large or complex parameter values.', ["my model scores 99% on training data but badly on new data, what's going on", 'जब मॉडल ट्रेनिंग डेटा का शोर याद कर लेता है तो उसे क्या कहते हैं?']),
    ('c03', 'L1 regularization, used in Lasso regression, adds the sum of the absolute values of the weights to the loss. Because of the shape of this penalty, many weights are pushed exactly to zero, so Lasso also performs automatic feature selection.', ['which penalty makes some coefficients become exactly zero', 'regularization method that also picks features for you']),
    ('c04', 'L2 regularization, used in Ridge regression and called weight decay in neural networks, adds the sum of squared weights to the loss. It shrinks all weights smoothly toward zero but rarely makes them exactly zero, which helps when many features are correlated.', ['what is weight decay in neural nets', 'which regularization shrinks coefficients but keeps all features, good for correlated inputs']),
    ('c05', 'In k-fold cross-validation the dataset is split into k equal parts. The model is trained k times, each time holding out a different part for evaluation, and the k scores are averaged. This gives a more reliable estimate of performance than a single train/test split, especially on small datasets.', ['how to get a more reliable accuracy estimate when I only have a small dataset', '¿cómo funciona la validación cruzada con k particiones?']),
    ('c06', 'Data should be split into training, validation and test sets before any preprocessing is fitted. If statistics such as the mean used for scaling are computed on the full dataset, information from the test set leaks into training and the reported score becomes too optimistic. This problem is called data leakage.', ['why is my test score unrealistically high, I scaled the data before splitting', 'should I fit the scaler on all the data or only on the training set']),
    ('c07', 'Gradient descent minimizes a loss function by repeatedly moving the parameters a small step in the direction opposite to the gradient. The step size is the learning rate: if it is too small, training is very slow; if it is too large, the loss can bounce around or diverge.', ['my loss keeps exploding during training, what setting should I lower', 'how does the basic optimization algorithm for training models work']),
    ('c08', 'Batch gradient descent uses the whole dataset for every update, which is slow for large data. Stochastic gradient descent updates after each single example, and mini-batch gradient descent uses small groups such as 32 or 256 examples. Mini-batches are the standard choice because they balance noisy updates with efficient use of GPUs.', ['why do people train with batches of 32 instead of the full dataset', 'difference between batch, stochastic and mini-batch updates']),
    ('c09', 'Adam is an adaptive optimizer that keeps a running average of past gradients (momentum) and of their squared values. It uses these to give each parameter its own effective learning rate, which usually makes training converge quickly with little tuning. It is a common default optimizer for deep learning.', ['which optimizer should I use by default for a neural network', 'optimizer that gives every weight its own learning rate using momentum']),
    ('c10', 'Linear regression predicts a numeric target as a weighted sum of input features plus an intercept. The weights are usually found by ordinary least squares, which chooses the line or plane that minimizes the sum of squared differences between predicted and actual values.', ['simplest model to predict house prices from size and number of rooms', 'what does ordinary least squares minimize']),
    ('c11', 'Logistic regression is used for binary classification. It passes a linear combination of the features through the sigmoid function, which squashes any number into a probability between 0 and 1. It is trained by minimizing log loss, also called binary cross-entropy.', ['which classic model outputs a probability for yes/no problems', 'what does the sigmoid function do']),
    ('c12', 'Precision is the fraction of predicted positives that are actually positive, while recall is the fraction of actual positives that the model found. In spam filtering you may favor precision so real emails are not lost; in disease screening you favor recall so few sick patients are missed. The F1 score is the harmonic mean of the two.', ['for detecting cancer, what matters more: catching every case or avoiding false alarms', 'what is F1 score']),
    ('c13', 'The ROC curve plots the true positive rate against the false positive rate as the classification threshold changes. The area under this curve, AUC, summarizes performance across all thresholds: 0.5 means random guessing and 1.0 means perfect separation of the classes.', ['what does an AUC of 0.5 mean', 'como avaliar um classificador em todos os limiares de decisão?']),
    ('c14', "Accuracy can be misleading when classes are imbalanced. If only 1% of transactions are fraud, a model that always predicts 'not fraud' scores 99% accuracy while being useless. A confusion matrix, which counts true positives, false positives, true negatives and false negatives, shows what is really happening.", ['my fraud model has 99 percent accuracy but catches nothing, why', 'table showing true and false positives and negatives']),
    ('c15', 'To handle imbalanced classes you can oversample the minority class, undersample the majority class, or create synthetic minority examples with SMOTE. Another option is to keep the data as is and give a higher class weight to the rare class in the loss function.', ['how to fix a dataset where one class is very rare', 'what is SMOTE']),
    ('c16', 'A decision tree splits the data with a sequence of yes/no questions on the features. At each node it chooses the split that makes the child groups as pure as possible, measured by Gini impurity or entropy. Trees are easy to interpret but grow deep and overfit unless their depth is limited or they are pruned.', ['model that makes predictions using a flowchart of if-else questions', 'what is Gini impurity used for']),
    ('c17', 'A random forest trains many decision trees on bootstrap samples of the data, a technique called bagging, and also considers only a random subset of features at each split. Averaging the votes of these diverse trees reduces variance and makes the forest far more robust than a single tree.', ['how does bagging reduce overfitting of trees', 'why is a forest of trees better than one tree']),
    ('c18', 'Gradient boosting builds trees sequentially: each new small tree is trained to correct the errors, or residuals, of the ensemble built so far. Libraries such as XGBoost, LightGBM and CatBoost implement this efficiently and are often the strongest choice for tabular data competitions.', ['best algorithm for Kaggle tabular data', 'ensemble where each new tree fixes the mistakes of the previous ones']),
    ('c19', 'A support vector machine finds the boundary that separates two classes with the widest possible margin. Only the points closest to the boundary, the support vectors, define it. With the kernel trick, an SVM can learn curved boundaries by implicitly mapping data into a higher-dimensional space.', ['classifier that maximizes the gap between classes', "how can a linear classifier separate data that isn't linearly separable"]),
    ('c20', 'The k-nearest neighbors algorithm has no real training step. To classify a new point it finds the k most similar points in the training data and takes a majority vote of their labels. It is simple but becomes slow on large datasets and is sensitive to the scale of the features.', ['lazy learning algorithm that looks at the closest examples to decide', 'which algorithm just votes among similar samples']),
    ('c21', "Naive Bayes classifiers apply Bayes' theorem while assuming that features are independent of each other given the class. The assumption is rarely true, yet the method works surprisingly well for text tasks such as spam detection and is extremely fast to train.", ['fast simple classifier for spam emails based on word probabilities', 'why is this Bayes classifier called naive']),
    ('c22', 'K-means clustering groups unlabeled data into k clusters. It starts with k centroids, assigns each point to the nearest centroid, moves each centroid to the mean of its assigned points, and repeats until assignments stop changing. The elbow method helps choose a reasonable k.', ['how to group customers into segments without labels', 'how do I choose the number of clusters']),
    ('c23', 'Principal component analysis reduces the number of features by projecting the data onto new axes, the principal components, that capture the most variance. The first component points in the direction where the data varies most. PCA is used to compress data, remove noise and speed up other algorithms.', ['how to reduce 100 features to 10 while keeping most of the information', 'técnica para reduzir a dimensionalidade dos dados mantendo a variância']),
    ('c24', 't-SNE and UMAP are non-linear techniques for visualizing high-dimensional data in two or three dimensions. They try to keep points that are close in the original space close in the plot. They are great for exploring clusters, but distances between far-apart groups in the plot should not be over-interpreted.', ['how to make a 2D picture of my embeddings to see clusters', 'is the distance between clusters in a t-SNE plot meaningful']),
    ('c25', 'Many algorithms, including gradient descent, k-nearest neighbors and SVMs, work poorly when features are on very different scales. Standardization rescales a feature to mean 0 and standard deviation 1, while min-max normalization squeezes it into the range 0 to 1. Tree-based models generally do not need scaling.', ['do I need to normalize features for a random forest', 'one feature is in thousands and another in decimals, is that a problem']),
    ('c26', 'Most models need numbers, so categorical features such as city or color must be encoded. One-hot encoding creates a separate 0/1 column for every category. When a feature has thousands of categories, alternatives such as target encoding or learned embeddings are more practical.', ['how to feed a column like country name into a model', 'what to do with a categorical feature that has thousands of values']),
    ('c27', 'Real datasets often have missing values. Simple strategies are dropping rows or filling gaps with the mean, median or most frequent value. More advanced imputation predicts the missing value from other features, and adding a flag column that marks where data was missing can itself be informative.', ['some cells in my dataset are empty, how should I handle them', '¿con qué valor relleno los datos faltantes?']),
    ('c28', 'An artificial neuron computes a weighted sum of its inputs and passes it through a non-linear activation function. Without non-linearity, stacking layers would still produce only a linear model. ReLU, which outputs zero for negative inputs and the input itself otherwise, is the most common activation in hidden layers.', ['why do neural networks need activation functions', 'what does ReLU do']),
    ('c29', 'Backpropagation computes the gradient of the loss with respect to every weight in a neural network. It runs a forward pass to get predictions, then applies the chain rule backwards from the output layer to the input layer. These gradients are then used by an optimizer to update the weights.', ['how does a neural network know how to adjust each weight', 'chain rule in training deep nets']),
    ('c30', 'In very deep networks, gradients can shrink as they are multiplied through many layers, so early layers barely learn; this is the vanishing gradient problem. Sigmoid activations make it worse. ReLU activations, careful weight initialization, residual connections and normalization layers all help.', ['why do the first layers of my deep network stop learning', 'what problem do residual connections solve']),
    ('c31', 'Dropout is a regularization technique for neural networks. During training it randomly switches off a fraction of neurons, for example 20 to 50 percent, at each step. This prevents neurons from relying too much on each other. At inference time all neurons are active.', ['randomly turning off neurons during training, what is this called', 'is dropout used at prediction time']),
    ('c32', 'Batch normalization normalizes the activations of a layer using the mean and variance of the current mini-batch, then applies a learned scale and shift. It stabilizes and speeds up training and allows higher learning rates. Layer normalization, used in transformers, normalizes across the features of a single example instead.', ['difference between batch norm and layer norm', 'technique that normalizes layer outputs to speed up training']),
    ('c33', 'Convolutional neural networks are designed for images. Convolution layers slide small learnable filters over the image to detect local patterns such as edges and textures, and pooling layers shrink the feature maps. Deeper layers combine simple patterns into complex shapes like eyes or wheels.', ['what type of network is used for image classification', 'what do the filters learn in the early layers']),
    ('c34', 'Recurrent neural networks process sequences one step at a time while carrying a hidden state that summarizes what they have seen. Plain RNNs struggle with long sequences, so LSTM and GRU cells add gates that decide what information to keep or forget over many time steps.', ['neural network with memory for time series', 'why were LSTMs invented']),
    ('c35', 'The transformer architecture replaces recurrence with self-attention: every token computes how much to attend to every other token, so relationships across a whole sentence are captured in parallel. Positional encodings add word order. Transformers are the foundation of modern language models such as BERT and GPT.', ['what architecture is behind GPT', '¿cómo funciona la autoatención en los transformers?']),
    ('c36', 'Word embeddings represent words as dense vectors so that words used in similar contexts get similar vectors. Word2vec learns them by predicting a word from its neighbors or the neighbors from the word. The famous example is that king minus man plus woman lands close to queen.', ['king minus man plus woman equals queen', 'how are words turned into vectors']),
    ('c37', 'Transfer learning reuses a model pretrained on a large dataset for a new task. You typically freeze the early layers, which capture general features, and fine-tune the later layers on your smaller dataset. This gives strong results with far less data and compute than training from scratch.', ['I only have 500 labeled images, can I still train a good model', 'fine-tuning a pretrained network']),
    ('c38', 'Early stopping monitors the validation loss during training and stops when it has not improved for a set number of epochs, called the patience. The weights from the best epoch are restored. It is a simple and effective guard against overfitting in iterative training.', ['when should I stop training, how many epochs', 'validation loss started going up while training loss keeps going down']),
    ('c39', 'Hyperparameters, such as learning rate or tree depth, are set before training rather than learned. Grid search tries every combination on a fixed grid, random search samples combinations and is often more efficient, and Bayesian optimization tools like Optuna use past results to choose promising settings next.', ['how to find the best learning rate and tree depth automatically', 'grid search vs random search']),
    ('c40', 'As the number of features grows, the volume of the space grows so fast that data points become sparse and far apart. Distance-based methods lose meaning because all points look almost equally distant. This curse of dimensionality is why feature selection and dimensionality reduction matter.', ['why does kNN get worse with hundreds of features', 'what happens to distances in very high dimensional spaces']),
    ('c41', 'In reinforcement learning an agent interacts with an environment, takes actions, and receives rewards. Its goal is to learn a policy that maximizes total future reward. Q-learning estimates the value of each action in each state, and the agent must balance exploring new actions with exploiting known good ones.', ['how do AIs learn to play games through trial and error', 'exploration vs exploitation']),
    ('c42', 'A generative adversarial network trains two networks against each other. The generator creates fake samples from random noise, and the discriminator tries to tell fake from real. As both improve, the generator learns to produce realistic images. Training can be unstable, and mode collapse is a common failure.', ['two networks competing, one creates fakes and one detects them', 'what is mode collapse']),
    ('c43', 'An autoencoder compresses its input into a small latent representation with an encoder and reconstructs it with a decoder. It is used for dimensionality reduction, denoising and anomaly detection, since unusual inputs reconstruct poorly. A variational autoencoder makes the latent space smooth so new samples can be generated.', ['detect anomalies using reconstruction error', 'what is a VAE']),
    ('c44', 'Regression models are evaluated with error metrics. Mean absolute error is the average size of mistakes, while root mean squared error penalizes large mistakes more heavily. R-squared shows the share of variance in the target explained by the model, where 1 is a perfect fit.', ['RMSE vs MAE, which one should I report', 'what does an R squared of 0.8 mean']),
    ('c45', 'For multi-class classification, the final layer produces one score per class, and the softmax function turns these scores into probabilities that sum to 1. The model is trained with categorical cross-entropy loss, which heavily penalizes confident wrong predictions.', ['how does a network output probabilities for 10 classes', 'loss function for classifying digits 0-9']),
    ('c46', 'Data augmentation creates new training examples by applying label-preserving changes to existing ones. For images this includes flips, rotations, crops and color changes; for text it includes synonym replacement or back-translation. It effectively enlarges the dataset and improves generalization.', ['how to get more training images without collecting new data', 'back-translation for text data']),
    ('c47', "After deployment, a model's performance can decay because the incoming data changes over time, which is known as data drift or concept drift. Production systems should monitor input distributions and prediction quality, and retrain the model when significant drift is detected.", ['my model was great at launch but got worse months later', 'मॉडल को दोबारा कब ट्रेन करना चाहिए?']),
    ('c48', "Maximum likelihood estimation picks the parameters that make the observed data most probable. Maximum a posteriori estimation also multiplies in a prior belief about the parameters using Bayes' theorem. With a Gaussian prior, MAP estimation turns out to be equivalent to L2 regularization.", ['difference between MLE and MAP', 'connection between priors and regularization']),
    ('c49', 'Learning curves plot training and validation error as the training set grows. If both errors are high and close together, the model underfits and needs more capacity. If training error is low but validation error stays much higher, the model overfits and more data or regularization will help.', ['how to tell if more data will help my model', 'plot of error versus training set size']),
    ('c50', 'Retrieval-augmented generation combines a language model with a search step. Documents are split into chunks, converted to embeddings and stored in a vector database. At question time the most similar chunks are retrieved and given to the model as context, which reduces hallucination and lets it answer from private data.', ['how can a chatbot answer questions from my company documents', 'why store document chunks in a vector database']),
]

TITLES = {'c01': 'Bias-variance tradeoff', 'c02': 'Overfitting', 'c03': 'L1 regularization (Lasso)', 'c04': 'L2 regularization (Ridge)', 'c05': 'Cross-validation', 'c06': 'Data leakage', 'c07': 'Gradient descent', 'c08': 'Mini-batch training', 'c09': 'Adam optimizer', 'c10': 'Linear regression', 'c11': 'Logistic regression', 'c12': 'Precision and recall', 'c13': 'ROC curve and AUC', 'c14': 'Accuracy and the confusion matrix', 'c15': 'Imbalanced classes', 'c16': 'Decision trees', 'c17': 'Random forests', 'c18': 'Gradient boosting', 'c19': 'Support vector machines', 'c20': 'k-nearest neighbors', 'c21': 'Naive Bayes', 'c22': 'K-means clustering', 'c23': 'Principal component analysis', 'c24': 't-SNE and UMAP', 'c25': 'Feature scaling', 'c26': 'Categorical encoding', 'c27': 'Missing values', 'c28': 'Neurons and activation functions', 'c29': 'Backpropagation', 'c30': 'Vanishing gradients', 'c31': 'Dropout', 'c32': 'Batch normalization', 'c33': 'Convolutional networks', 'c34': 'Recurrent networks and LSTMs', 'c35': 'Transformers and attention', 'c36': 'Word embeddings', 'c37': 'Transfer learning', 'c38': 'Early stopping', 'c39': 'Hyperparameter tuning', 'c40': 'Curse of dimensionality', 'c41': 'Reinforcement learning', 'c42': 'Generative adversarial networks', 'c43': 'Autoencoders', 'c44': 'Regression metrics', 'c45': 'Softmax and cross-entropy', 'c46': 'Data augmentation', 'c47': 'Model drift', 'c48': 'MLE and MAP', 'c49': 'Learning curves', 'c50': 'Retrieval-augmented generation'}

CHAPTERS = [
    ('Chapter 1: Foundations', ['c01', 'c02', 'c03', 'c04', 'c38', 'c49', 'c40', 'c48']),
    ('Chapter 2: Evaluation', ['c05', 'c06', 'c12', 'c13', 'c14', 'c15', 'c44']),
    ('Chapter 3: Classic algorithms', ['c10', 'c11', 'c16', 'c17', 'c18', 'c19', 'c20', 'c21']),
    ('Chapter 4: Unsupervised learning', ['c22', 'c23', 'c24', 'c43']),
    ('Chapter 5: Data preparation', ['c25', 'c26', 'c27', 'c46']),
    ('Chapter 6: Optimization', ['c07', 'c08', 'c09', 'c39']),
    ('Chapter 7: Neural networks', ['c28', 'c29', 'c30', 'c31', 'c32', 'c33', 'c34', 'c35', 'c36', 'c37', 'c45', 'c42']),
    ('Chapter 8: Beyond supervised learning', ['c41', 'c47', 'c50']),
]

os.makedirs("docs", exist_ok=True)
by_id = {cid: text for cid, text, _ in DATA}
lines = ["# Machine Learning Handbook", ""]
for chapter, ids in CHAPTERS:
    lines += [f"## {chapter}", ""]
    for cid in ids:
        lines += [f"### {TITLES[cid]}", "", by_id[cid], ""]
with open("docs/ml_handbook.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

def pick_answer(passage, query):
    """Use the passage sentence that best matches the question as the answer text."""
    sents = re.split(r"(?<=[.!?])\s+", passage)
    qw = {w for w in re.findall(r"\w+", query.lower()) if len(w) > 3}
    overlap = lambda s: len(qw & set(re.findall(r"\w+", s.lower())))
    best = max(sents, key=overlap)
    return best if overlap(best) > 0 else sents[0]

rows = [{"query": q, "answer_text": pick_answer(text, q), "source": "ml_handbook.md"}
        for _, text, qs in DATA for q in qs]
pd.DataFrame(rows).to_csv("eval.csv", index=False)
print(f"Demo created: docs/ml_handbook.md ({len(DATA)} sections), eval.csv ({len(rows)} questions)")

## 2. Load documents and questions

In [ ]:
import os, pandas as pd
from pypdf import PdfReader

def read_doc(path):
    if path.lower().endswith(".pdf"):
        return "\n\n".join((p.extract_text() or "") for p in PdfReader(path).pages)
    return open(path, encoding="utf-8", errors="ignore").read()

docs = [{"name": f, "text": read_doc(os.path.join("docs", f))}
        for f in sorted(os.listdir("docs")) if f.lower().endswith((".md", ".txt", ".pdf"))]
evalset = pd.read_csv("eval.csv").dropna(subset=["query", "answer_text"]).reset_index(drop=True)
print(f"{len(docs)} documents, {sum(len(d['text']) for d in docs):,} characters, {len(evalset)} questions")

## 3. Settings
`TOKEN_BUDGET` is how much retrieved text you'd send to the LLM. Keep it the same across strategies so the comparison is fair.

In [ ]:
EMBED_MODEL  = "microsoft/harrier-oss-v1-0.6b"
QUERY_PROMPT = "web_search_query"   # "query" for Qwen3-Embedding, None for BGE-M3
USE_RERANKER = False                # True = rerank the top candidates with the model below
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
TOKEN_BUDGET = 1500                 # how many context tokens you would send to the LLM
CANDIDATES   = 20                   # results kept per question (also what the reranker sees)

## 4. Load the embedding model (and reranker if enabled)

In [ ]:
import torch, numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = {"torch_dtype": torch.float16} if DEVICE == "cuda" else {}

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE, model_kwargs=DTYPE)
embedder.max_seq_length = 1024
tok = embedder.tokenizer
reranker = CrossEncoder(RERANK_MODEL, max_length=512, device=DEVICE, model_kwargs=DTYPE) if USE_RERANKER else None

def ntok(text):
    return len(tok(text, add_special_tokens=False)["input_ids"])

def embed_docs(texts):
    return embedder.encode(texts, batch_size=16, normalize_embeddings=True, show_progress_bar=False)

def embed_queries(texts):
    kw = {"prompt_name": QUERY_PROMPT} if QUERY_PROMPT else {}
    return embedder.encode(texts, batch_size=16, normalize_embeddings=True, **kw)

def rerank_scores(query, texts):
    return reranker.predict([(query, t) for t in texts], batch_size=16)

print("Loaded on", DEVICE)

## 5. The chunking strategies
| Strategy | How it cuts |
|---|---|
| `fixed_tokens` | Every N tokens, ignoring sentences and headings (baseline) |
| `recursive` | Paragraphs, then sentences, then words, up to N tokens, with overlap |
| `structure` | Markdown sections first, then recursive inside each; optional heading path added to every chunk |
| `semantic` | Starts a new chunk where the meaning between neighbouring sentences jumps |
| `parent_child` | Searches small chunks but returns their larger parent section |

In [ ]:
import re
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

SEPARATORS = ["\n\n", "\n", ". ", "? ", "! ", " ", ""]
HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3")]

def recursive_splitter(size, overlap):
    return RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tok, chunk_size=size, chunk_overlap=overlap, separators=SEPARATORS)

def unit(text, source, group=None, search=None):
    # search = text that gets embedded; text = what is returned to the LLM and graded
    return {"search": search or text, "text": text, "source": source, "group": group}

def sections(doc, levels=3):
    md = MarkdownHeaderTextSplitter(HEADERS[:levels], strip_headers=False)
    for s in md.split_text(doc["text"]):
        path = " > ".join(s.metadata[k] for _, k in HEADERS[:levels] if k in s.metadata)
        yield s.page_content, path

def fixed_tokens(size, overlap=0):
    """Cut every N tokens, ignoring sentences and headings."""
    def run(docs):
        out = []
        for d in docs:
            ids = tok(d["text"], add_special_tokens=False)["input_ids"]
            for i in range(0, len(ids), size - overlap):
                out.append(unit(tok.decode(ids[i:i + size]), d["name"]))
                if i + size >= len(ids):
                    break
        return out
    return run

def recursive(size, overlap):
    """Split on paragraphs, then sentences, then words, up to N tokens."""
    def run(docs):
        sp = recursive_splitter(size, overlap)
        return [unit(c, d["name"]) for d in docs for c in sp.split_text(d["text"])]
    return run

def structure(size, overlap, add_context=False):
    """Split on markdown headings first, then recursively inside each section."""
    def run(docs):
        sp, out = recursive_splitter(size, overlap), []
        for d in docs:
            for body, path in sections(d):
                for c in sp.split_text(body):
                    prefix = f"{d['name']} > {path}\n" if add_context and path else ""
                    out.append(unit(prefix + c, d["name"]))
        return out
    return run

def semantic(max_tokens=400, percentile=90):
    """Start a new chunk where the meaning between two sentences jumps."""
    def run(docs):
        out = []
        for d in docs:
            sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n{2,}", d["text"]) if s.strip()]
            if len(sents) < 2:
                out += [unit(s, d["name"]) for s in sents]
                continue
            E = embed_docs(sents)
            gaps = 1 - (E[:-1] * E[1:]).sum(axis=1)
            threshold = np.percentile(gaps, percentile)
            cur, cur_tok = [sents[0]], ntok(sents[0])
            for s, gap in zip(sents[1:], gaps):
                t = ntok(s)
                if gap > threshold or cur_tok + t > max_tokens:
                    out.append(unit(" ".join(cur), d["name"]))
                    cur, cur_tok = [], 0
                cur.append(s)
                cur_tok += t
            out.append(unit(" ".join(cur), d["name"]))
        return out
    return run

def parent_child(child_size=150, child_overlap=30, parent_size=1000):
    """Search small child chunks, but return their larger parent section."""
    def run(docs):
        psp, csp = recursive_splitter(parent_size, 0), recursive_splitter(child_size, child_overlap)
        out, pid = [], 0
        for d in docs:
            for body, _ in sections(d, levels=2):
                for parent in psp.split_text(body):
                    for child in csp.split_text(parent):
                        out.append(unit(parent, d["name"], group=pid, search=child))
                    pid += 1
        return out
    return run

## 6. Evaluation
**How to read the metrics:**
- **Hit@budget** (main metric): add results in ranked order until the token budget is full; did the answer make it in? This is fair across sizes, because big chunks use up the budget faster.
- **Hit@1 / 3 / 5, MRR@10**: same as the embedding test. Careful: bigger chunks look better here simply because each chunk contains more text.
- **top5_context_tokens**: how much text 5 results cost you in the LLM prompt.
- **over_512_%**: share of chunks too long for bge-reranker-v2-m3's 512-token input (with the question). The reranker would never see the end of those.
- **chunks_searched / chunks_returned**: for parent-child, many small chunks are searched but fewer parents are returned.

In [ ]:
import time
from rapidfuzz import fuzz

def norm(s):
    return re.sub(r"[^\w]+", " ", str(s).lower()).strip()

def contains(answer, text):
    """True if the returned text contains the answer (exactly, or 90%+ fuzzy match)."""
    a, t = norm(answer), norm(text)
    if a in t:
        return True
    return len(t) >= 0.9 * len(a) and fuzz.partial_ratio(a, t) >= 90

_q_emb = None

def evaluate(name, chunker):
    global _q_emb
    if _q_emb is None:
        _q_emb = embed_queries(evalset["query"].astype(str).tolist())
    t0 = time.time()
    units = chunker(docs)
    U = embed_docs([u["search"] for u in units])
    secs = time.time() - t0
    utok = [ntok(u["text"]) for u in units]
    key = lambda i: units[i]["group"] if units[i]["group"] is not None else f"u{i}"
    sims = _q_emb @ U.T

    hits, mrr, hit_budget, ctx5, misses = {1: 0, 3: 0, 5: 0}, 0.0, 0, 0, []
    for qi, (q, ans) in enumerate(zip(evalset["query"].astype(str), evalset["answer_text"].astype(str))):
        ranked, seen = [], set()
        for ui in np.argsort(-sims[qi]):
            if key(ui) in seen:
                continue
            seen.add(key(ui))
            ranked.append(int(ui))
            if len(ranked) == CANDIDATES:
                break
        if reranker is not None:
            rs = rerank_scores(q, [units[i]["text"] for i in ranked])
            ranked = [ranked[i] for i in np.argsort(-rs)]

        found = [contains(ans, units[i]["text"]) for i in ranked]
        first = found.index(True) if True in found else None
        for k in hits:
            hits[k] += first is not None and first < k
        if first is not None and first < 10:
            mrr += 1 / (first + 1)
        used = 0
        for i, f in zip(ranked, found):
            if used and used + utok[i] > TOKEN_BUDGET:
                break
            used += utok[i]
            if f:
                hit_budget += 1
                break
        ctx5 += sum(utok[i] for i in ranked[:5])
        if first is None or first >= 5:
            misses.append((q, ans, units[ranked[0]]["text"]))

    n = len(evalset)
    returned = {key(i): utok[i] for i in range(len(units))}
    toks = list(returned.values())
    return {
        "strategy": name,
        f"Hit@budget({TOKEN_BUDGET})": round(hit_budget / n, 3),
        "Hit@1": round(hits[1] / n, 3),
        "Hit@3": round(hits[3] / n, 3),
        "Hit@5": round(hits[5] / n, 3),
        "MRR@10": round(mrr / n, 3),
        "chunks_searched": len(units),
        "chunks_returned": len(toks),
        "avg_tokens": round(float(np.mean(toks)), 1),
        "top5_context_tokens": round(ctx5 / n),
        "over_512_%": round(100 * float(np.mean([t + 32 > 512 for t in toks])), 1),
        "seconds": round(secs, 1),
    }, misses

## 7. Run all strategies
Edit the dictionary to add sizes or remove strategies.

In [ ]:
STRATEGIES = {
    "fixed 256 tokens, no overlap":         fixed_tokens(256, 0),
    "fixed 512 tokens, 64 overlap":         fixed_tokens(512, 64),
    "recursive 300, 50 overlap":            recursive(300, 50),
    "recursive 500, 75 overlap":            recursive(500, 75),
    "structure-aware 400":                  structure(400, 50),
    "structure-aware 400 + context header": structure(400, 50, add_context=True),
    "semantic (p90, max 400)":              semantic(400, 90),
    "parent-child 150 -> 1000":             parent_child(150, 30, 1000),
}

results, all_misses = [], {}
for name, chunker in STRATEGIES.items():
    print("running:", name)
    res, misses = evaluate(name, chunker)
    results.append(res)
    all_misses[name] = misses

report = pd.DataFrame(results).sort_values(f"Hit@budget({TOKEN_BUDGET})", ascending=False).reset_index(drop=True)
report

## 8. Where each strategy failed
Read these. They show whether answers were cut in half at a boundary, drowned in a big chunk, or retrieved from the wrong section.

In [ ]:
for name, misses in all_misses.items():
    print(f"\n=== {name}: {len(misses)} questions without the answer in the top 5 ===")
    for q, ans, top in misses[:5]:
        print(f"- Q: {q}\n  expected: {ans[:100]}\n  got #1:   {top[:100]!r}")

## 9. (Optional) Look at the chunks a strategy produces

In [ ]:
STRATEGY_TO_VIEW = "structure-aware 400"
for u in STRATEGIES[STRATEGY_TO_VIEW](docs)[:4]:
    print(f"[{ntok(u['text'])} tokens]\n{u['text'][:500]}\n" + "-" * 60)

## How to decide
1. Sort by **Hit@budget**. Shortlist the top 2–3.
2. Among those, prefer fewer **top5_context_tokens** (cheaper, less *lost in the middle*) and **over_512_%** near 0 if you use the reranker.
3. Set `USE_RERANKER = True`, re-run, and check the winner still wins with reranking on, since that's your real pipeline.
4. Check the misses of the winner before committing.